In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives']


In [3]:
from lib import beatport
mio   = beatport.MusicDBIO(verbose=False)
webio = beatport.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant Beatport DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Beatport


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists         = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localArtistsReleases = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsReleases".format(db.lower()))
localArtistsTracks   = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsTracks".format(db.lower()))
masterArtists        = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
searchArtists        = mio.data.getSearchArtistData()
knownArtists         = mio.data.getSummaryNameData()
errors               = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Artists:          {0}".format(len(localArtists.get())))
print("   Local Artists Releases: {0}".format(len(localArtistsReleases.get())))
print("   Local Artists Tracks:   {0}".format(len(localArtistsTracks.get())))
print("   Master Artists:         {0}".format(len(masterArtists.get())))
print("   Errors:                 {0}".format(len(errors.get())))
print("   Search Artists:         {0}".format(searchArtists.shape[0]))
print("   Known IDs:              {0}".format(len(knownArtists)))

Beatport Search Results
   Local Artists:          203325
   Local Artists Releases: 22852
   Local Artists Tracks:   0
   Master Artists:         0
   Errors:                 80
   Search Artists:         203405
   Known IDs:              203187


# Download Artist Data

In [ ]:
mio   = beatport.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = beatport.RawWebData(debug=False)

In [ ]:
useSearchData = True

if useSearchData is True:
    artistNames      = searchArtists.sort_values(by="Num", ascending=False)
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())].rename(columns={"Ref": "URL"})

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(localArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  39998
#   Artist Names To Get:  33523
#   Artist Names To Get:  12448

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "10:00am")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["Name"]
    artistURL  = row["URL"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistURL=artistURL)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(4.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

In [ ]:
localArtists.save(data=localArtistsDict)

In [ ]:
from lib.beatport import moveLocalFiles
moveLocalFiles()

In [ ]:
from utils import PoolIO
pio = PoolIO("Beatport")
pio.parse()
pio.merge()
pio.meta()
pio.sum()
pio.search()

# Download Artist Releases Data

In [6]:
mio   = beatport.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = beatport.RawWebData(debug=False)

In [59]:
useCountsData = True

if useCountsData is True:
    artistNamesX = DataFrame(mio.data.getSummaryNameData()).join(mio.data.getSummaryRefData()).join(mio.data.getSummaryCountsData())[["Name", "Ref", "Album"]].sort_values(by="Album", ascending=False)
    artistNames = artistNamesX[artistNamesX["Album"] > 0]
    localArtistsReleasesDict = localArtistsReleases.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsReleasesDict.keys())].rename(columns={"Ref": "URL"})

    print("# {0} Search Results".format(db))
    print("#   Available Names:      {0}".format(len(artistNamesX)))
    print("#     At Least One Album: {0}".format(len(artistNames)))
    print("#   Known Artist Names:   {0}".format(len(localArtistsReleasesDict)))
    print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  115669
#   Artist Names To Get:  113069
#   Artist Names To Get:  109244
#   Artist Names To Get:  106869

# Beatport Search Results
#   Available Names:      203187
#     At Least One Album: 202771
#   Known Artist Names:   98852
#   Artist Names To Get:  103919


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "6:50am")
tt = TermTime("today", "8:00pm")
maxN = 50000000

n  = 0
localArtistsReleasesDict = localArtistsReleases.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["Name"]
    artistURL  = row["URL"]
    if localArtistsReleasesDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistReleaseData(artistName=artistName, artistURL=artistURL)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(4)
        continue
        
    mio.data.saveRawArtistReleaseData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsReleasesDict[artistID] = artistName
    webio.sleep(4.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsReleasesDict), db))
        localArtistsReleases.save(data=localArtistsReleasesDict)
        print("="*150)
        webio.wait(10.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsReleasesDict), db))
localArtistsReleases.save(data=localArtistsReleasesDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting Beatport ArtistIDs] Start    ==> Time Is 2022-05-23 07:07:56
========================= termTime(day=today,time=8:00pm) =========================
   ====> Terminate Time Set To 2022-05-23 20:00:00 <====
   ====> Will Terminate Process 12 Hours and 52 Minutes From Now
Getting Data For Mr. Calix (679407)                                True
Getting Data For Claudia Zatara (744907)                           True
Getting Data For thatmanmonkz (292807)                             True
Getting Data For Loyal (592107)                                    True
Getting Data For Vastive (570607)                                  True
Getting Data For NJ Smart (764907)                                 True
Getting Data For Venus In Motion (79007)                           True
Getting Data For Malone (94807)                                    True
Getting Data For Mr. Tophat (318807)                               True
Getting Data For Shiba San (375907)                                T

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting Data For Maetrik (2707)                                    True
Getting Data For Oraa (147107)                                     True
Getting Data For Tragedy Khadafi (48107)                           True
Getting Data For TRYLOW (870307)                                   True
Getting Data For Samadhi (167307)                                  True
Getting Data For Zove (608307)                                     True
Getting Data For The Machine (25007)                               True
Getting Data For Egapia (760607)                                   True
Getting Data For Counterweight (127707)                            True
Getting Data For Davide Marchesiello (151107)                      True
Getting Data For RoneeDeep (671807)                                True
Getting Data For Steve Mill (24607)                                True
Getting Data For Antennasia (53407)                                True
Getting Data For Chris Rockz (108807)                           

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting Data For A-Tronix (565807)                                 True
Getting Data For Bon (19607)                                       True
Getting Data For Beats Control (359807)                            True
Getting Data For Norman (9607)                                     True
Getting Data For Molto Mojto (709907)                              True
Getting Data For Bass Bumpers (8207)                               True
Getting Data For Imene Bella (771407)                              True
Getting Data For Babbooyn (772807)                                 True
Getting Data For Danny Voxmann (455307)                            True
Getting Data For Future Funk (35607)                               True
Getting Data For Andrea T Mendoza (22207)                          True
Getting Data For Dark Flesher (696707)                             True
Getting Data For Darrel Peterson (746207)                          True
Getting Data For Decibel Artforce (271807)                      

...

Getting Data For M4LY (715110)                                     True
Getting Data For Paco Caniza (500810)                              True
Getting Data For Last Japan (105110)                               True
Getting Data For Marvin Aloys (216410)                             True
Getting Data For Maydie Myles (645810)                             True
Getting Data For Dinamo Azari (99310)                              True
Getting Data For Alan Hash (379610)                                True
Getting Data For Funk Machine (36710)                              True
Getting Data For Lars Laroc (549110)                               True
Getting Data For Sebastian Brandt (15310)                          True
Getting Data For Kaori (96610)                                     True
Getting Data For Angus McDonald (803010)                           True
Getting Data For Rod Hayden (529910)                               True
Getting Data For The Audio Manipulator (694910)                 

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting Data For Acid Kids (41310)                                 True
Getting Data For Frank Walker (420210)                             True
Getting Data For Sunnery James & Ryan Marciano (76410)             True
Getting Data For Jaksan (504110)                                   True
Getting Data For Will Diamond (118410)                             True
Getting Data For DNCN (20210)                                      True
Getting Data For Twist3d (175410)                                  True
Getting Data For Gaetano Visconti (699110)                         True
Getting Data For Unlighted (786610)                                True
Getting Data For Qt 8 (173310)                                     True
Getting Data For Ashley Wallbridge (64910)                         True
Getting Data For Jon Hanley (211809)                               True
Getting Data For Mathey B (212210)                                 True
Getting Data For Miss P (184310)                                

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Getting Data For Oona Dahl (160110)                                True
Getting Data For Soul Entity (580510)                              True
Getting Data For Matu (300610)                                     True
Getting Data For Sarah Lynn (141310)                               True
Getting Data For Dirty Freek (63610)                               True
Getting Data For Boy Matthews (526510)                             True


# Download Artist Tracks Data

In [ ]:
useCountsData = True

if useCountsData is True:
    artistNamesX = DataFrame(mio.data.getSummaryNameData()).join(mio.data.getSummaryRefData()).join(mio.data.getSummaryCountsData())[["Name", "Ref", "SingleEP"]].sort_values(by="SingleEP", ascending=False)
    artistNames = artistNamesX[artistNamesX["SingleEP"] > 0]
    localArtistsReleasesDict = localArtistsReleases.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsReleasesDict.keys())].rename(columns={"Ref": "URL"})

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNamesX)))
    print("     At Least One Track: {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(localArtistsReleasesDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  39998
#   Artist Names To Get:  33523
#   Artist Names To Get:  12448

# Create Starter

In [ ]:
from collections import Counter
from ioutils import FileIO, HTMLIO
io = FileIO()
hio = HTMLIO()
cntr = Counter()
refs = []
for modVal in range(100):
    for ifile in mio.dir.getRawModValDataDir(modVal).glob("*.p"):
        bsdata = hio.get(io.get(ifile))
        refs  += [(ref.attrs['data-artist'],ref.attrs['href'],ref.text) for ref in bsdata.findAll('a') if ref.attrs.get('data-artist') is not None]
    if (modVal+1) % 5 == 0:
        print(modVal,'\t',len(refs))

In [ ]:
df = DataFrame(refs)
N  = df[0].value_counts()
N.name = "Num"
df = df.drop_duplicates()
df.index = df[0]
df.index.name = None
df = df.drop([0], axis=1)
df.columns = ["Ref", "Name"]
df = df.join(N)
df.sort_values(by='Num', ascending=False)

In [ ]:
mio.data.saveSearchArtistData(data=df)